In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors



In [ ]:
# 2. Create TensorDataset objects

from torchvision.datasets import CIFAR10
from torchvision.transforms.functional import to_tensor
train_dataset = CIFAR10(
    root='./datasets',
    train=True,
    transform=to_tensor,
    download=True
)

test_dataset = CIFAR10(
    root='./datasets',
    train=False,
    transform=to_tensor,
    download=True
)


In [ ]:
# 3. Create DataLoaders
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# 4. Print shape of one batch
import matplotlib.pyplot as plt
images, labels = next(iter(train_loader))

plt.figure(figsize=(8, 4))
for i in range(6):
    plt.subplot(2, 3, i + 1)


In [ ]:
# 5. Display sample images
images, labels = next(iter(train_loader))

plt.figure(figsize=(8, 4))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    img = images[i].permute(1, 2, 0)
    plt.imshow(img)
    plt.title(f"Label: {labels[i].item()}")
    plt.axis('off')
plt.show()


In [ ]:
# Task 1: Write your model class here:
import torch.nn as nn
class NN4Layer(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(NN5Layer, self).__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)
        self.layer4 = nn.Linear(hidden_dim, hidden_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        z1 = self.layer1(x)
        a1 = self.relu(z1)

        z2 = self.layer2(a1)
        a2 = self.relu(z2)

        z3 = self.layer3(a2)
        a3 = self.relu(z3)

        z4 = self.layer4(a3)
        a4 = self.relu(z4)


In [ ]:
# Task 2: Write your training loop here:
import torch.nn.functional as F
def train_one_epoch(model, optimizer, criterion, train_loader, device):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.view(X_batch.size(0), -1).to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(train_loader)


In [ ]:
# Task 3: Write your validation loop here:
def validate(model, criterion, test_loader, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.view(X_batch.size(0), -1).to(device)
            y_batch = y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            running_loss += loss.item()

            probabilities = F.softmax(outputs, dim=1)
            predicted = torch.argmax(probabilities, dim=1)

            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

    accuracy = correct / total
    return running_loss / len(test_loader), accuracy


In [ ]:
# Task 4: Define device, model, loss, optimizer:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_dim = 3 * 32 * 32
hidden_dim = 64
output_dim = 10

model = NN4Layer(input_dim, hidden_dim, output_dim).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=0.001)
num_epochs = 15



In [ ]:
# Task 5: Start training for 20 epochs:
from torchvision.datasets import CIFAR10
from torchvision.transforms.functional import to_tensor
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import torch
from torch.optim import AdamW
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)
    val_loss, val_accuracy = validate(model, criterion, test_loader, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_accuracy:.4f}")


In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2 (Bonus): Write your code here: